# import

We want to implement here the constrained rotation of the agent around a single axis, to mimic the absence of the infomration processing in 3D

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append("../General_Functions/")
from BasicData import *
from Flow_functions import * # import flow functions, Information to the agent
from Geometric_functions import * # Import rotation from euler angles
from BasicData import *  # Nagents, R0, V
from Strategies import * # Import strategies, Braitenberg or Triangulation
from Strategies_Evolution import * # Import strategies Evolution, Braitenberg or Triangulation
from Flow_functions import * # Import flow functions: stokeslet, grads, strains ... 


In [19]:
# Integrate numerically the angular dynamics of each strategy
# Calculate the scalar product between t at the end of the evolution
# and the real source position

def Evolution(Grad):

    # Function to evaluate the scalar product between t at the end of the evolution
    # and the real source position
    #input:
    # Gradient (source flow measured)
    #output:
    # dot product as a 2D map
    # variance of dot product as 2D map
    
    # t axis for Triangulation strategy absence of roll
    t0_T, t1_T, t2_T = np.zeros((NPoints, NPoints, NTrials)), np.zeros((NPoints, NPoints, NTrials)), np.zeros((NPoints, NPoints, NTrials))
    t0_T_1D, t1_T_1D, t2_T_1D = np.zeros((NPoints, NPoints, NTrials)), np.zeros((NPoints, NPoints, NTrials)), np.zeros((NPoints, NPoints, NTrials))
 
    # Resulting dot products
    DotProduct_T = np.zeros((NPoints, NPoints, NTrials))
    DotProduct_T_1D = np.zeros((NPoints, NPoints, NTrials))

    for k in range(NPoints):
        for i in range(NPoints):
            # Position the agent 
            AgentPosition =  np.array([z[k], x[i], 0.])

            # Source position
            OptimalDirection = -AgentPosition/np.linalg.norm(AgentPosition)

            for trials_i in range(NTrials):
                # Random orientation of agent axis
                alpha, beta, gamma = (2 * np.random.rand(3) - 1) * np.pi
                RotationMatrix = Rotation(alpha, np.abs(beta), gamma)

                n_T = np.dot(RotationMatrix,[0, 1, 0])
                b_T = np.dot(RotationMatrix,[0, 0, 1])
                t_T = np.dot(RotationMatrix,[1, 0, 0])

                # 1D version 
                n_T_1D = np.dot(RotationMatrix,[0, 1, 0])
                b_T_1D = np.dot(RotationMatrix,[0, 0, 1])
                t_T_1D = np.dot(RotationMatrix,[1, 0, 0])
                
                # Time integration of strategies
                for time in range(Duration):
                    n_T_1D, b_T_1D, t_T_1D, _ = Evolution_Triangulation_1D(n_T_1D, b_T_1D, t_T_1D, AntennaeLength, AgentPosition, Grad, e3, dt, 0, roll)
                    n_T, b_T, t_T, _ = Evolution_Triangulation(n_T, b_T, t_T, AntennaeLength, AgentPosition, Grad, e3, dt, 0, roll)
                   
                t0_T[k, i, trials_i], t1_T[k, i, trials_i], t2_T[k, i, trials_i] = t_T
                t0_T_1D[k, i, trials_i], t1_T_1D[k, i, trials_i], t2_T_1D[k, i, trials_i] = t_T_1D
               
                # Dot product Evaluation
                DotProduct_T[k, i, trials_i] = np.dot(t_T, OptimalDirection)
                DotProduct_T_1D[k, i, trials_i] = np.dot(t_T_1D, OptimalDirection)
               

    return DotProduct_T, \
        t0_T, t1_T, \
        DotProduct_T_1D , \
        t0_T_1D , t1_T_1D 
       

In [24]:
# Data
NPoints = 50 # Grid number of points per axis
Duration = 255 # Number of timesteps of the numerical integration
NTrials = 100 # Number of trial x point

AntennaeLength = .01 # Sensors distance, on antennae
x = np.linspace(0.1, 5, NPoints) # x axis
z = np.linspace(0.1, 5, NPoints) # z axis

e3 = np.array([1, 0, 0]) # Source flow axis of symmetry, scheme zxy
dt = .1 # Timesteps 
roll = .3 # Roll Intensity
NameStrategies = ["T"] #"T",

In [ ]:
# 

# Information flow 
GradName = ['GradientStresslet'] # 'ShearQuadruplet', 'GradientQuadruplet', 
Grads =  [GradientStresslet] # ShearQuadruplet, GradientQuadruplet


for grads_i, Grad in enumerate(Grads):
    # t axis for Triangulation strategy absence of roll
    t0_T, t1_T = np.zeros((NPoints, NPoints, NTrials)), np.zeros((NPoints, NPoints, NTrials))
    t0_T_1D, t1_T_1D = np.zeros((NPoints, NPoints, NTrials)), np.zeros((NPoints, NPoints, NTrials))

    # Evaluate the strategy 2D map, vector P 
    Avg_P_T = np.zeros((NPoints, NPoints, NTrials))
    Avg_P_T_1D = np.zeros((NPoints, NPoints, NTrials))

    # Call Evaluation function and evaluate 
    Avg_P_T, t0_T, t1_T, Avg_P_T_1D, t0_T_1D, t1_T_1D = Evolution(Grad)
    
    # Save data
    AllAvg = [Avg_P_T] #, Avg_P_T, Avg_P_I]
    Allt0 = [t0_T] # _not, t0_T, t0_I]
    Allt1 = [t1_T] # _not, t1_T, t1_I]
    
    AllAvg_1D = [Avg_P_T_1D] #, Avg_P_T, Avg_P_I]
    Allt0_1D = [t0_T_1D] # _not, t0_T, t0_I]
    Allt1_1D = [t1_T_1D] # _not, t1_T, t1_I]

    for i, strat_name in enumerate(NameStrategies):
        np.savetxt("Data_1D_roll/"+str(GradName[grads_i])+"_Avg_"+str(strat_name)+"_test.txt", np.mean(AllAvg[i], axis = 2), delimiter = ',')
        np.savetxt("Data_1D_roll/"+str(GradName[grads_i])+"_t0_"+str(strat_name)+"_test.txt", np.mean(Allt0[i], axis = 2), delimiter = ',')
        np.savetxt("Data_1D_roll/"+str(GradName[grads_i])+"_t1_"+str(strat_name)+"_test.txt", np.mean(Allt1[i], axis = 2), delimiter = ',')
        
        np.savetxt("Data_1D_roll/"+str(GradName[grads_i])+"_Avg_"+str(strat_name)+"_test_1D.txt", np.mean(AllAvg_1D[i], axis = 2), delimiter = ',')
        np.savetxt("Data_1D_roll/"+str(GradName[grads_i])+"_t0_"+str(strat_name)+"_test_1D.txt", np.mean(Allt0_1D[i], axis = 2), delimiter = ',')
        np.savetxt("Data_1D_roll/"+str(GradName[grads_i])+"_t1_"+str(strat_name)+"_test_1D.txt", np.mean(Allt1_1D[i], axis = 2), delimiter = ',')


#plt.show()

In [ ]:
# Plot data from Save 
# Information flow 

import scipy.ndimage as ndimage # import to smooth plots
GradName = ['GradientStresslet'] # 'ShearQuadruplet', 'GradientQuadruplet', 
Grads =  [GradientStresslet] # ShearQuadruplet, GradientQuadruplet

for grads_i, Grad in enumerate(GradName):
    for i, strat_name in enumerate(NameStrategies):
        
        # Extract data from saved file
        with open("Data_1D_roll/"+str(GradName[grads_i])+"_Avg_"+str(strat_name)+"_test.txt", 'r') as the_file:
            Avg = np.array([np.asarray(each_line.split(",")).astype(float) for each_line in the_file]) 

        with open("Data_1D_roll/"+str(GradName[grads_i])+"_Avg_"+str(strat_name)+"_test_1D.txt", 'r') as the_file:
            Avg_1D = np.array([np.asarray(each_line.split(",")).astype(float) for each_line in the_file])
            
        with open("Data_1D_roll/"+str(GradName[grads_i])+"_t0_"+str(strat_name)+"_test.txt", 'r') as the_file:
            t0 = np.array([np.asarray(each_line.split(",")).astype(float) for each_line in the_file]) 

        with open("Data_1D_roll/"+str(GradName[grads_i])+"_t0_"+str(strat_name)+"_test_1D.txt", 'r') as the_file:
            t0_1D = np.array([np.asarray(each_line.split(",")).astype(float) for each_line in the_file])
            
        with open("Data_1D_roll/"+str(GradName[grads_i])+"_t1_"+str(strat_name)+"_test.txt", 'r') as the_file:
            t1 = np.array([np.asarray(each_line.split(",")).astype(float) for each_line in the_file]) 

        with open("Data_1D_roll/"+str(GradName[grads_i])+"_t1_"+str(strat_name)+"_test_1D.txt", 'r') as the_file:
            t1_1D = np.array([np.asarray(each_line.split(",")).astype(float) for each_line in the_file]) 

        
        # Plot parameters
        CMAP = 'RdBu_r'  # 'spring'
        STREAMCOLOR = 'black'
        MIN_VALUE = -1
        MAX_VALUE = 1
        
        # Create 1 row, 2 columns of subplots
        fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(16,8), sharex=True, sharey=True)
        
        X, Z = np.meshgrid(x, z, indexing='xy')
        
        # Common contour levels
        levels = np.linspace(-1, 1.00001, 14)
        
        # --- First subplot ---
        Avg_smooth = ndimage.gaussian_filter(Avg, sigma=1.0)
        im0 = axs[0].contourf(X, Z, Avg, cmap=CMAP, levels=levels, vmin=MIN_VALUE, vmax=MAX_VALUE, alpha=0.5)
        axs[0].streamplot(X, Z, t1, t0, color=STREAMCOLOR, density=0.3, arrowsize=2.5, broken_streamlines=False)
        axs[0].set_xlim([0.1, 5])
        axs[0].set_ylim([0.1, 5])
        axs[0].set_xticks([])
        axs[0].set_yticks([])
        axs[0].set_title('complete')
        
        # --- Second subplot ---
        # Example using the same data, replace Avg2, t1_2, t0_2 with your second dataset
        Avg2_smooth = ndimage.gaussian_filter(Avg, sigma=1.0)
        im1 = axs[1].contourf(X, Z, Avg_1D, cmap=CMAP, levels=levels, vmin=MIN_VALUE, vmax=MAX_VALUE, alpha=0.5)
        axs[1].streamplot(X, Z, t1_1D, t0_1D, color=STREAMCOLOR, density=0.3, arrowsize=2.5, broken_streamlines=False)
        axs[1].set_xlim([0.1, 5])
        axs[1].set_ylim([0.1, 5])
        axs[1].set_xticks([])
        axs[1].set_yticks([])
        axs[1].set_title('1D')
        
        # Add a shared colorbar
        cbar = fig.colorbar(im0, ax=axs, orientation='horizontal', fraction=0.05, pad=0.1, ticks=np.arange(-1, 1.1, 0.5))
        cbar.set_label('Value')
        
        # Save the figure
        plt.savefig(f"Plots_1D_roll/{GradName[grads_i]}_{strat_name}_Numerical.eps", format='eps', bbox_inches='tight')
        plt.savefig(f"Plots_1D_roll/{GradName[grads_i]}_{strat_name}_Numerical.pdf", bbox_inches='tight')
        plt.show()